In [ ]:
makeData = False

In [ ]:
# ==============================================================================
# Part 1: Installation
# ==============================================================================
# This cell installs the required libs.
# ------------------------------------------------------------------------------

# !pip install -q synapseclient "monai[nibabel, tqdm]" joblib scikit-image

import subprocess
import sys
import pkg_resources

def install_if_not_installed(package):
    try:
        # Check if package is installed
        pkg_resources.get_distribution(package)
        print(f"{package} is already installed.")
    except pkg_resources.DistributionNotFound:
        print(f"{package} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# List of required packages
packages = [
    "synapseclient",
    "monai[nibabel,tqdm]",
    "joblib",
    "scikit-image"
]

for package in packages:
    install_if_not_installed(package)

In [ ]:
# ==============================================================================
# Part 2: UserSecretsClient
# ==============================================================================
# Import Kaggle UserSecretsClient for secure token access ---
# ------------------------------------------------------------------------------

if makeData : 
    print("No")

import os
try:
    # Try to load Synapse token securely from Kaggle
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    stk = user_secrets.get_secret("SYNAPSE_AUTH_TOKEN")
    print("Synapse token loaded successfully from Kaggle Secrets.")
except ImportError:
    # Fallback for local execution if not in Kaggle
    stk = os.environ.get("SYNAPSE_AUTH_TOKEN", None)
    if stk:
        print("Synapse token loaded successfully from environment variable.")
    else:
        # Fallback for local execution if secrets/env var not set.
        # Replace "PASTE_YOUR_TOKEN_HERE" with your actual token for local tests.
        stk = "PASTE_YOUR_TOKEN_HERE"
        if stk == "PASTE_YOUR_TOKEN_HERE":
            raise RuntimeError("Synapse token not found. Please set it in Kaggle Secrets (key: SYNAPSE_AUTH_TOKEN) or define the 'stk' variable manually.")
        else:
            print("Synapse token loaded from manual variable definition.")

In [ ]:
# ==============================================================================
# Part 2: Data Downloading & Unzipping
# ==============================================================================
# This section handles authentication and download of the dataset from Synapse.
# ------------------------------------------------------------------------------
import synapseclient
import zipfile
import shutil

if makeData : 
    print("No")

# --- Synapse Login ---
syn = synapseclient.Synapse()
try:
    syn.login(authToken=stk, silent=True)
    print("Synapse login successful.")
except Exception as e:
    print(f"Synapse login failed. Please ensure your authToken is correct. Error: {e}")
    raise

# --- File & Directory Setup ---
idc = ["syn51514132"]
destination_dir = '/kaggle/working/BRATS/train'
os.makedirs(destination_dir, exist_ok=True)

# --- Helper Functions for Download & Unzip ---
def unzip_data(zip_path, extract_to):
    print(f"Checking for unzipped data at {extract_to}...")
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 50:
        print("Data appears to be already unzipped. Skipping.")
        return
    print(f"Unzipping {zip_path} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Unzipping complete.")
    
    # Handle nested folders
    content_list = os.listdir(extract_to)
    if len(content_list) == 1 and os.path.isdir(os.path.join(extract_to, content_list[0])):
        nested_folder = os.path.join(extract_to, content_list[0])
        print(f"Moving contents from nested folder '{nested_folder}'...")
        for item in os.listdir(nested_folder):
            shutil.move(os.path.join(nested_folder, item), extract_to)
        os.rmdir(nested_folder)

# --- Download & Unzip Execution ---
unzipped_path = os.path.join('/kaggle/working/BRATS/train')
if os.path.exists(unzipped_path) and len(os.listdir(unzipped_path)) > 50:
    print("Dataset already downloaded and unzipped. Skipping download.")
else:
    print(f"--- Starting Download: {idc[0]} ---")
    dat = syn.get(entity=idc[0])
    unzip_data(dat.path, destination_dir)
    os.remove(dat.path)
    print(f"Deleted zip file: {dat.path}")
print("--- Data Preparation Finished ---")



In [ ]:
# ==============================================================================
# Part 3: Preprocessing (NIfTI to NPZ)
# ==============================================================================
# This optimized pipeline converts raw .nii.gz files into processed .npz arrays.
# ------------------------------------------------------------------------------

if makeData : 
    print("No")

import numpy as np
from tqdm.notebook import tqdm
import concurrent.futures
from functools import partial
import torch

from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, CropForegroundd,
    Resized, ScaleIntensityRanged, ConvertToMultiChannelBasedOnBratsClassesd, EnsureTyped
)

# --- Configuration ---
BASE_DIR = '/kaggle/working/BRATS'
UNZIPPED_DIR = os.path.join(BASE_DIR, 'train')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed')

TARGET_VOXEL_SPACING = (1.0, 1.0, 1.0)       # Resample voxel spacing
OUTPUT_SHAPE = (128, 128, 128)               # Final 3D volume shape
MODALITY_KEYS = ['t1c', 't1n', 't2f', 't2w'] # MRI modalities
ALL_KEYS = MODALITY_KEYS + ['seg']           # Include segmentation label

os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Preprocessing Functions ---
# --- Find patient files ---
def find_patient_files(data_dir):
    patient_files = []
    for patient_id in sorted(os.listdir(data_dir)):
        patient_folder = os.path.join(data_dir, patient_id)
        if os.path.isdir(patient_folder):
            files = {
                "t1c": os.path.join(patient_folder, f"{patient_id}-t1c.nii.gz"),
                "t1n": os.path.join(patient_folder, f"{patient_id}-t1n.nii.gz"),
                "t2f": os.path.join(patient_folder, f"{patient_id}-t2f.nii.gz"),
                "t2w": os.path.join(patient_folder, f"{patient_id}-t2w.nii.gz"),
                "seg": os.path.join(patient_folder, f"{patient_id}-seg.nii.gz"),
                "id": patient_id
            }
            # Add only if all modality files exist
            if all(os.path.exists(f) for k, f in files.items() if k != "id"):
                patient_files.append(files)
    return patient_files

# --- Define MONAI preprocessing pipeline ---
monai_preprocess_pipeline = Compose([
    LoadImaged(keys=ALL_KEYS, image_only=True, ensure_channel_first=True),  # Load .nii.gz files
    ConvertToMultiChannelBasedOnBratsClassesd(keys='seg'),  # Convert seg labels into 3 channels
    Spacingd(keys=ALL_KEYS, pixdim=TARGET_VOXEL_SPACING, mode=["bilinear"] * 4 + ["nearest"]),  # Resample
    ScaleIntensityRanged(keys=MODALITY_KEYS, a_min=0.0, a_max=1400.0, b_min=0.0, b_max=1.0, clip=True), # Normalize intensity
    CropForegroundd(keys=ALL_KEYS, source_key='t1c', margin=10),  # Crop region around tumor
    Resized(keys=ALL_KEYS, spatial_size=OUTPUT_SHAPE, mode=["area"] * 4 + ["nearest"]),  # Resize to uniform shape
    EnsureTyped(keys=ALL_KEYS, dtype=torch.float16) # Use float16 to save disk space
])

# --- Function: process and save one patient ---
def preprocess_and_save(patient_data, output_dir):
    try:
        processed_data = monai_preprocess_pipeline(patient_data)
         # Concatenate all modality images into one tensor
        final_image = torch.cat([processed_data[key] for key in MODALITY_KEYS], dim=0)
        final_mask = processed_data['seg']  # Segmentation mask
        output_filepath = os.path.join(output_dir, f"{patient_data['id']}.npz")
        np.savez_compressed(output_filepath, image=final_image.numpy(), mask=final_mask.numpy().astype(np.uint8))
    except Exception as e:
        return f"Failed {patient_data['id']}: {e}"
    return None

# --- Main Preprocessing Execution ---
# --- Run preprocessing in parallel ---
patient_list = find_patient_files(UNZIPPED_DIR)
if not os.listdir(PROCESSED_DIR):     # Skip if already processed
    print(f"Found {len(patient_list)} patients to preprocess.")
    process_func = partial(preprocess_and_save, output_dir=PROCESSED_DIR)
    with concurrent.futures.ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
        results = list(tqdm(executor.map(process_func, patient_list), total=len(patient_list), desc="Preprocessing"))
    print("\nPreprocessing complete!")
else:
    print("Processed data already exists. Skipping preprocessing.")

In [ ]:
# ==============================================================================
# Part 4: Data Integrity Check & K-Fold Split
# ==============================================================================

from sklearn.model_selection import KFold

if makeData : 
    print("No")

# --- Collect valid processed files (non-empty .npz only) ---
VALID_FILES = sorted([os.path.join(PROCESSED_DIR, f) for f in os.listdir(PROCESSED_DIR) if f.endswith('.npz') and os.path.getsize(os.path.join(PROCESSED_DIR, f)) > 0])
print(f"\nFound {len(VALID_FILES)} valid patient files.")

# --- K-Fold parameters ---
N_SPLITS = 5              # 5 folds for cross-validation
FOLD_TO_RUN = 0           # Currently at zero/no fold can take (0-4)
RANDOM_STATE = 42         # Seed for reproducibility

# --- Split dataset into training and validation sets ---
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
train_indices, val_indices = list(kf.split(VALID_FILES))[FOLD_TO_RUN]
train_files = [VALID_FILES[i] for i in train_indices]
val_files = [VALID_FILES[i] for i in val_indices]

# --- Print dataset sizes ---
print(f"\n--- FOLD {FOLD_TO_RUN}/{N_SPLITS-1} ---")
print(f"Training set size: {len(train_files)} | Validation set size: {len(val_files)}")



In [ ]:
# ==============================================================================
# Part 5: Dataset and DataLoaders
# ==============================================================================

from torch.utils.data import Dataset, DataLoader
from monai.transforms import Compose, RandFlipd, RandRotate90d, Rand3DElasticd, RandScaleIntensityd

# --- Custom Dataset Class for BraTS ---
class BraTSDataset(Dataset):
    def __init__(self, file_paths, augment=False):
        self.file_paths = file_paths             # List of .npz file paths
        self.augment = augment                   # Enable data augmentation if True
        
        # Define augmentation pipeline if augment=True
        if self.augment:
            self.transform = Compose([
                RandFlipd(keys=["image", "mask"], prob=0.5, spatial_axis=0),  # Random flip (X)
                RandFlipd(keys=["image", "mask"], prob=0.5, spatial_axis=1),  # Random flip (Y)
                RandFlipd(keys=["image", "mask"], prob=0.5, spatial_axis=2),  # Random flip (Z)
                RandRotate90d(keys=["image", "mask"], prob=0.5, max_k=3),     # Random rotation 90°
                RandScaleIntensityd(keys="image", factors=0.1, prob=0.5),     # Intensity scaling
                Rand3DElasticd(keys=["image", "mask"], sigma_range=(3, 5), magnitude_range=(10, 30),
                               prob=0.2, mode=('bilinear', 'nearest'), padding_mode='zeros')   # Elastic deformation
            ])

    def __len__(self):
        return len(self.file_paths)   # Number of samples

    def __getitem__(self, idx):
        # Load image and mask tensors from .npz file
        with np.load(self.file_paths[idx]) as data:
            image = torch.from_numpy(data['image'].astype(np.float32))
            mask = torch.from_numpy(data['mask'].astype(np.float32))
            
        # Apply augmentation if enabled
        if self.augment:
            data_dict = self.transform({"image": image, "mask": mask})
            return data_dict["image"], data_dict["mask"]
        return image, mask    # Return as (image, mask) pair

# --- Create datasets ---
train_dataset = BraTSDataset(file_paths=train_files, augment=True)
val_dataset = BraTSDataset(file_paths=val_files, augment=False)

# --- Define DataLoader parameters ---
BATCH_SIZE = 2
NUM_WORKERS = 2 # Set to 0 for debugging if you get DataLoader errors

# --- Create DataLoaders for training & validation ---
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0))


In [ ]:
# ==============================================================================
# Part 6: Model, Loss, Optimizer, and Scheduler
# ==============================================================================
# Switched to AttentionUnet with safer channel sizes for Kaggle Compute
# Model's forward pass does not have a final sigmoid activation ---
# ------------------------------------------------------------------------------

import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from monai.networks.nets import AttentionUnet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- MODIFIED: Reduced initial channels to prevent OOM errors. Increase if memory permits. ---
model = AttentionUnet(
    spatial_dims=3, in_channels=4, out_channels=3,
    channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
).to(device)

# --- Define Dice Loss ---
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
        
    def forward(self, logits, targets):
        logits, targets = logits.view(-1), targets.view(-1)    # Flatten
        intersection = (logits * targets).sum()                # Intersection
        sum_of_sets = logits.sum() + targets.sum()             # Union
        dice_coeff = (2. * intersection + self.smooth) / (sum_of_sets + self.smooth)
        return 1 - dice_coeff                                  # Return Dice loss (1 - Dice)

# --- Combine Dice + BCE Loss ---
class DiceBCELoss(nn.Module):
    def __init__(self, weight_dice=0.5, weight_bce=0.5):
        super().__init__()
        self.dice_loss = DiceLoss()
        self.bce_loss = nn.BCEWithLogitsLoss()                # Combines Sigmoid + BCE
        self.weight_dice = weight_dice
        self.weight_bce = weight_bce
        
    def forward(self, logits, targets):
        dice = self.dice_loss(torch.sigmoid(logits), targets)  # Dice on sigmoid outputs
        bce = self.bce_loss(logits, targets)                   # BCE on raw logits
        return self.weight_dice * dice + self.weight_bce * bce # Weighted combination

# --- Training Hyperparameters ---
LEARNING_RATE = 1e-4
NUM_EPOCHS = 20
PATIENCE = 7    # For early stopping

# --- Initialize criterion, optimizer, and scheduler ---
criterion = DiceBCELoss().to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)  # AdamW for stable training

# Polynomial LR decay schedule
poly_lambda = lambda epoch: (1 - epoch / NUM_EPOCHS) ** 0.9
scheduler = LambdaLR(optimizer, lr_lambda=poly_lambda)

# --- Use AMP for mixed-precision training (faster + less memory) ---
scaler = torch.cuda.amp.GradScaler()


In [ ]:
# ==============================================================================
# Part 7: Training & Validation Loop with AMP and Early Stopping
# ==============================================================================
# Integrated AMP, scheduler, early stopping, and detailed metrics
# Checkpointing now saves a full dictionary for resuming
# ------------------------------------------------------------------------------

import time

BEST_MODEL_PATH = f'/kaggle/working/best_model_fold_{FOLD_TO_RUN}.pth'

# --- Function to compute Dice per class ---
def dice_score_per_class(preds, targets, smooth=1e-6):
    preds = torch.sigmoid(preds) > 0.5       # Convert logits to binary masks
    dice_scores = []
    for i in range(preds.shape[1]):          # Iterate over channels (classes)
        pred_flat = preds[:, i, ...].contiguous().view(-1)
        target_flat = targets[:, i, ...].contiguous().view(-1)
        intersection = (pred_flat * target_flat).sum()
        union = pred_flat.sum() + target_flat.sum()
        dice = (2. * intersection + smooth) / (union + smooth)
        dice_scores.append(dice)
    return torch.stack(dice_scores)           # Tensor of Dice scores per class

# --- One training epoch ---
def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(loader, desc="Training", leave=False)
    for images, masks in progress_bar:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():            # Enable mixed precision
            outputs = model(images)                # Forward pass
            loss = criterion(outputs, masks)       # Compute loss
        scaler.scale(loss).backward()              # Backprop scaled loss
        scaler.step(optimizer)                     # Update weights
        scaler.update()                            # Update scaler
        running_loss += loss.item()
    return running_loss / len(loader)              # Average training loss

# --- One validation epoch ---
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_dice_scores = []
    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Validating", leave=False):
            images, masks = images.to(device), masks.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
            dice = dice_score_per_class(outputs.cpu(), masks.cpu())    # Dice per class
            all_dice_scores.append(dice)
            running_loss += loss.item()
    
    avg_dice = torch.stack(all_dice_scores).mean(0)                    # Mean Dice per class
    return running_loss / len(loader), avg_dice

# --- Main Training Loop ---
best_val_dice = -1.0
no_improve_epochs = 0
start_time = time.time()

print(f"\nStarting Training for Fold {FOLD_TO_RUN}...")
for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_dice_per_class = validate_one_epoch(model, val_loader, criterion, device)
    scheduler.step()           # Adjust learning rate
    
    avg_val_dice = val_dice_per_class.mean().item()          # Mean Dice across all classes
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Dice (Avg): {avg_val_dice:.4f} [WT: {val_dice_per_class[0]:.4f}, TC: {val_dice_per_class[1]:.4f}, ET: {val_dice_per_class[2]:.4f}]")

    # --- Save best model ---
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        no_improve_epochs = 0
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'best_val_dice': best_val_dice
        }, BEST_MODEL_PATH)
        print(f"✅ New best model saved (Dice: {best_val_dice:.4f})")
    else:
        no_improve_epochs += 1

    # --- Early stopping check ---
    if no_improve_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1} (no improvement in {PATIENCE} epochs).")
        break

print(f"\n✅ Training Finished in {(time.time() - start_time) / 60:.2f} minutes.")


